In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, expr, window
)
from pyspark.sql.types import LongType, IntegerType



Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: org.apache.spark.SparkException: Invalid Spark URL: spark://HeartbeatReceiver@BUI_DUY:60620
	at org.apache.spark.rpc.RpcEndpointAddress$.apply(RpcEndpointAddress.scala:66)
	at org.apache.spark.rpc.netty.NettyRpcEnv.asyncSetupEndpointRefByURI(NettyRpcEnv.scala:143)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.executor.Executor.<init>(Executor.scala:313)
	at org.apache.spark.scheduler.local.LocalEndpoint.<init>(LocalSchedulerBackend.scala:66)
	at org.apache.spark.scheduler.local.LocalSchedulerBackend.start(LocalSchedulerBackend.scala:136)
	at org.apache.spark.scheduler.TaskSchedulerImpl.start(TaskSchedulerImpl.scala:237)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:622)
	at org.apache.

In [ ]:
SOURCE_TABLE = "dev.bronze.streaming_bronze"
TARGET_TABLE = "dev.silver.events_silver"

CHECKPOINT_PATH = "/Volumes/dev/silver/checkpoint/_checkpoints/silver_streaming_event/"


In [ ]:
%sql
select * from dev.bronze.streaming_bronze
limit 1

In [ ]:
# Preview bronze table (must exist in the catalog)
try:
    bronze = spark.table(SOURCE_TABLE)
    print('Bronze count:', bronze.count())
    display(bronze.limit(20))
except Exception as e:
    print('Failed to read events_bronze table. Create or ingest bronze first.', e)

In [ ]:
# Transform: validate fields, cast types, compute ingestion metadata
try:
    df = bronze
    # keep required fields only (example schema: timestamp, visitorid, event, itemid, transactionid)
    df = df.filter(F.col('timestamp').isNotNull() & F.col('visitorid').isNotNull() & F.col('event').isNotNull())
    df = df.withColumn('event_timestamp', F.col('timestamp').cast('long'))
    df = df.withColumn('item_id', F.col('itemid').cast('int'))
    df = df.withColumn('_ingestion_time', F.current_timestamp())
    df = df.withColumn('ingestion_latency_seconds', (F.col('_ingestion_time').cast('long') - (F.col('event_timestamp')/1000)).cast('double'))
    df = df.withColumn('event_date', F.to_date(F.from_unixtime(F.col('event_timestamp')/1000)))
    display(df.select('event_timestamp','visitorid','event','item_id','event_date','ingestion_latency_seconds').limit(20))
except NameError:
    print('`bronze` dataframe not available; run previous cell to load the table.')

In [ ]:
# Write the cleaned results to a Delta Silver table partitioned by event_date (exploratory append)
try:
    df.write.format('delta').mode('append').partitionBy('event_date').saveAsTable('events_silver')
    print('Wrote records to events_silver (append)')
except Exception as e:
    print('Write failed — ensure Delta and permissions available.', e)

In [ ]:
# Quick validation queries on the newly written silver table
try:
    display(spark.sql('SELECT event, COUNT(*) as cnt FROM events_silver GROUP BY event ORDER BY cnt DESC'))
    display(spark.sql('SELECT MAX(_ingestion_time) as last_ingest FROM events_silver'))
except Exception as e:
    print('Validation queries failed — ensure events_silver exists', e)

In [ ]:
%sql
select * from dev.bronze.streaming_bronze
limit 10

In [ ]:
df_typed = df_validated.select(
    col("event_datetime").alias("event_time"),
    col("visitorid").alias("visitor_id"),
    col("event").alias("event_type"),
    col("itemid").cast(IntegerType()).alias("item_id"),
    col("transactionid").alias("transaction_id"),
    col("_ingestion_id"),
    col("_ingestion_timestamp"),
    col("_processing_time").alias("processing_time")
)

In [ ]:
df_enriched = df_typed \
    .withColumn("event_date", expr("DATE(event_time)")) \
    .withColumn("event_hour", expr("HOUR(event_time)")) \
    .withColumn(
        "ingestion_latency_seconds",
        expr("CAST(UNIX_TIMESTAMP(processing_time) - UNIX_TIMESTAMP(event_time) AS INT)")
    )

display(df_enriched, checkpointLocation=CHECKPOINT_PATH + '/_display_preview')


In [ ]:
query = df_enriched.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH + 'data') \
    .option("mergeSchema", "true") \
    .partitionBy("event_date") \
    .toTable(TARGET_TABLE)


In [ ]:
query.status


In [ ]:
validation_df = spark.readStream.table(SOURCE_TABLE)

validation_stats = validation_df.withColumn(
    "event_datetime",
    to_timestamp(col("timestamp").cast(LongType()) / 1000)
).groupBy(
    window(col("event_datetime"), "5 minutes")
).count()

validation_query = validation_stats.writeStream \
    .format("console") \
    .option("truncate", "false") \
    .start()
